# Análisis de sentimiento en tuits para social listening

- Pipeline de social listening: recolección de tuits públicos, limpieza, etiquetado de sentimiento, representación de texto y comparación de clasificadores.
- Caso de estudio: conversación pública sobre "vacuna covid" durante 2021, recolectada en 2022.
- Artículo completo, con el diagrama del pipeline y las decisiones de ingeniería: https://fuzzyfrog.ai/es/ai-lab/proyectos/negocio/analisis-sentimiento-tuits-social-listening-covid/
- **Nota:** este notebook no contiene tuits reales, usuarios reales, ni credenciales de ninguna API. Los ejemplos de texto son ilustrativos y sintéticos. Ninguna credencial de acceso a Twitter/X debe colocarse nunca en texto plano en un notebook.
- **Nota de actualidad:** la recolección pública masiva de tuits que este proyecto usó en 2022 ya no es replicable de la misma forma. La plataforma, ahora X, restringió fuertemente el acceso libre a este tipo de datos desde 2023.


## Diagrama del pipeline

- Recolección de tuits públicos → limpieza y deduplicación → etiquetado heurístico de sentimiento → representación de texto (Word2Vec, Bag of Words, TF-IDF) → comparación de clasificadores → modelo elegido.
- La elección final no se basó solo en exactitud global, se revisó el desempeño por clase.
- Diagrama editable disponible en el artículo de la plataforma (liga arriba).


## Carga de datos

- Datos originales: tuits públicos recolectados en 2022 sobre "vacuna covid", periodo 2021-01-01 a 2021-12-31.
- Por privacidad de terceros, este notebook usa una **muestra sintética de ejemplo** con la misma estructura que el dataset original, en vez de tuits reales.
- Si vas a replicar este proyecto hoy, la recolección pública directa sobre X ya no funciona igual, revisa la nota de actualidad de la celda inicial.


In [ ]:
import pandas as pd
import numpy as np
import re

# Muestra sintética de ejemplo, con la misma estructura que el dataset original
# (Tweet, Tweet limpio, len, ID, Fecha, Fuente, Likes, RTs).
# Sustituye esto por tu propia fuente de datos si tienes acceso a una API vigente.
data_ejemplo = pd.DataFrame({
    "Tweet": [
        "@usuario_ejemplo Qué buena noticia sobre la vacuna, por fin algo de esperanza #vacunacovid",
        "No confío en esta vacuna, prefiero esperar más estudios http://ejemplo.com",
        "Hoy me tocó mi segunda dosis, todo normal, sin efectos raros",
        "El gobierno debería explicar mejor el plan de vacunación, hay mucha confusión",
        "Excelente iniciativa de las autoridades de salud con la campaña de vacunación",
    ],
})

def clean_tweet(tweet_text):
    temp = tweet_text.lower()
    temp = re.sub("@[A-Za-z0-9_]+", "", temp)
    temp = re.sub("#[A-Za-z0-9_]+", "", temp)
    temp = re.sub(r"http\S+", "", temp)
    temp = temp.strip()
    return temp

data_ejemplo["Tweet limpio"] = data_ejemplo["Tweet"].apply(clean_tweet)
data_ejemplo


## Explicación de datos

- El dataset original tenía 5.000 tuits recolectados, reducidos a 4.734 tras limpiar texto vacío y eliminar duplicados exactos del texto limpio.
- La limpieza elimina menciones, hashtags, URLs, y normaliza el texto a minúsculas antes de cualquier análisis posterior.


In [ ]:
# En el proyecto original:
# 5000 tuits recolectados -> 4997 tras eliminar texto vacío -> 4734 tras eliminar duplicados
resumen_limpieza = pd.DataFrame({
    "etapa": ["Recolectados", "Sin texto vacío", "Sin duplicados"],
    "cantidad": [5000, 4997, 4734],
})
resumen_limpieza


## Análisis de datos / EDA

- Antes de modelar, conviene ver la distribución de longitud de los tuits y confirmar el desbalance entre clases de sentimiento.
- En el proyecto original, la distribución de sentimiento resultante (tras el etiquetado heurístico) fue de aproximadamente 226 positivos, 312 negativos y 409 neutros en el set de prueba de 947 ejemplos.


In [ ]:
distribucion_test_original = pd.DataFrame({
    "clase": ["Positivo", "Negativo", "Neutro"],
    "soporte_test": [226, 312, 409],
})
distribucion_test_original["proporcion"] = (
    distribucion_test_original["soporte_test"] / distribucion_test_original["soporte_test"].sum()
).round(3)
distribucion_test_original


## Modelado

- **Etiquetado de sentimiento (heurístico):** cada tuit se traduce palabra por palabra al inglés y se etiqueta según el signo de la polaridad de TextBlob. No es una etiqueta validada por una persona, es una aproximación.
- **Representación de texto:** se exploraron Word2Vec, Bag of Words y TF-IDF. La clasificación final se hizo sobre Bag of Words.
- **Clasificadores comparados:** Regresión Logística, SVM, Random Forest y Naive Bayes, todos sobre las mismas features.


In [ ]:
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, accuracy_score

def analizar_sentimiento_heuristico(texto):
    # Nota: en el proyecto original la traducción se hacía palabra por palabra al inglés
    # antes de calcular la polaridad con TextBlob. Aquí se simplifica para el ejemplo.
    analisis = TextBlob(texto)
    if analisis.sentiment.polarity > 0:
        return "Positivo"
    elif analisis.sentiment.polarity == 0:
        return "Neutro"
    else:
        return "Negativo"

data_ejemplo["Sentimiento"] = data_ejemplo["Tweet limpio"].apply(analizar_sentimiento_heuristico)

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(data_ejemplo["Tweet limpio"]).toarray()
y = data_ejemplo["Sentimiento"]

# En el dataset real, este split se hizo sobre 4.734 tuits con 947 de prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Evaluación

- Resultados reales del proyecto original, comparando los cuatro clasificadores sobre Bag of Words (947 ejemplos de prueba):


In [ ]:
resultados_comparacion = pd.DataFrame({
    "modelo": ["Regresión Logística", "SVM", "Random Forest", "Naive Bayes"],
    "exactitud": [0.71, 0.61, 0.62, 0.56],
    "recall_positivo": [0.54, 0.19, 0.18, 0.46],
    "precision_positivo": [0.71, 0.73, 0.82, 0.41],
})
resultados_comparacion


In [ ]:
# La lectura correcta no es solo "¿cuál tiene mayor exactitud?"
# SVM y Random Forest tienen precisión alta en la clase Positivo, pero un recall muy bajo:
# casi nunca predicen esa clase, aunque cuando lo hacen suelen acertar.
# Regresión Logística logra el mejor balance entre exactitud global y recall en la clase minoritaria.
mejor_modelo = resultados_comparacion.sort_values("recall_positivo", ascending=False).iloc[0]
print("Modelo elegido:", mejor_modelo["modelo"])
print("Motivo: mejor equilibrio entre exactitud global y recall en la clase minoritaria (positivo)")


## Hallazgos principales

- La exactitud global no bastó para elegir el mejor modelo. SVM y Random Forest tenían precisión alta en la clase "positivo" pero un recall de apenas 0.19 y 0.18, prácticamente invisibles a esa clase. Regresión Logística, con una exactitud similar (0.71), tuvo un recall de 0.54 en la misma clase.
- El etiquetado de sentimiento no vino de anotación humana, vino de una heurística de traducción y polaridad. Es razonable para un proyecto académico, pero no equivale a una etiqueta validada por una persona, y así debe leerse cualquier resultado derivado.
- De las tres representaciones de texto exploradas, Bag of Words, la más simple, fue la que se usó en la clasificación final, un recordatorio de que la técnica más sofisticada disponible no siempre es la más apropiada para el problema.
- Este proyecto se hizo con la mayor ética posible dentro del entendimiento disponible en 2022. Hoy, además, ni la forma de recolectar estos datos ni el estándar esperado de manejo de datos públicos se quedaron iguales, vale la pena reconocerlo antes de replicar un proyecto similar.
